In [23]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

In [2]:
# Define directories
train_dir = '/home/alienblade/College Stuff/Academics/CAO/DA/archive/Master Folder/train'
test_dir = '/home/alienblade/College Stuff/Academics/CAO/DA/archive/Master Folder/test'
validation_dir = '/home/alienblade/College Stuff/Academics/CAO/DA/archive/Master Folder/valid'

In [3]:
# Image size and batch size
img_size = (224, 224)
batch_size = 32

# Data augmentation for the training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [4]:
# No data augmentation for the validation and test sets
validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

Found 1000 images belonging to 4 classes.
Found 36 images belonging to 4 classes.
Found 38 images belonging to 4 classes.


In [5]:
# Load MobileNetV2 pre-trained model without the top (fully connected) layers
base_model = VGG16(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

# Freeze the pre-trained layers
for layer in base_model.layers:
    layer.trainable = False

2023-11-23 20:43:47.798836: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-11-23 20:43:47.799247: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


9406464/9406464 [==============================] - 0s 0us/step


In [6]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')  # 4 classes: Angry, Sad, Happy, Other
])

In [7]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [17]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=15,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)

Epoch 1/15
31/31 [==============================] - 15s 504ms/step - loss: 0.2671 - accuracy: 0.9091 - val_loss: 2.1093 - val_accuracy: 0.3438
Epoch 2/15
31/31 [==============================] - 17s 557ms/step - loss: 0.2769 - accuracy: 0.9081 - val_loss: 2.1039 - val_accuracy: 0.3750
Epoch 3/15
31/31 [==============================] - 18s 566ms/step - loss: 0.2660 - accuracy: 0.9163 - val_loss: 2.0613 - val_accuracy: 0.4062
Epoch 4/15
31/31 [==============================] - 16s 515ms/step - loss: 0.2205 - accuracy: 0.9246 - val_loss: 2.3677 - val_accuracy: 0.3438
Epoch 5/15
31/31 [==============================] - 16s 517ms/step - loss: 0.2116 - accuracy: 0.9339 - val_loss: 2.1318 - val_accuracy: 0.4375
Epoch 6/15
31/31 [==============================] - 16s 535ms/step - loss: 0.2711 - accuracy: 0.8926 - val_loss: 2.3323 - val_accuracy: 0.3750
Epoch 7/15
31/31 [==============================] - 16s 516ms/step - loss: 0.2447 - accuracy: 0.9101 - val_loss: 1.8678 - val_accuracy: 0.4688

In [22]:
# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator, steps=test_generator.samples // batch_size)
print(f'Test accuracy: {test_acc}')

1/1 [==============================] - 0s 426ms/step - loss: 2.2733 - accuracy: 0.5000
Test accuracy: 0.5
